# LELA32052 Computational Linguistics Week 4

## N-gram language models

In this seminar we will examine n-gram language models. The first thing that we will do is to build a table of bigram counts.

In order to this I will need to introduce you to another data structure in Python - the dictionary.


### Dictionaries
In an earlier session you encountered the List. A second useful Python data structure is the Dictionary. This stores data in key and value pairs. There is a flexibility in the data types that can be keys and can be values, for example the former could be a string or an int. The latter could be a list or even another dictionary.

In [1]:
thisdict = {
  "brand": "Ford",
  "model": "Mustang",
  "year": 1964
}

In [2]:
print(thisdict)

{'brand': 'Ford', 'model': 'Mustang', 'year': 1964}


You can obtain the keys as a standalone list

In [3]:
thisdict.keys()

dict_keys(['brand', 'model', 'year'])

And the same for the values

In [4]:
thisdict.values()

dict_values(['Ford', 'Mustang', 1964])

One useful additional thing to consider is that there are different kinds of dictionaries in the Collections library. We will make use of one special kind of dictionary later - the default dictionary which returns a default value when asked for a missing key.

### For Loops

A commonly used tool in programming is the "for loop". In its simplest form this allows us to iterate over (move through) a series of values. For example:

In [5]:
for i in range(0,10):
  print(i)

0
1
2
3
4
5
6
7
8
9


It can also be used to iterate through data structures:


In [6]:
sentence=["the","boy","went","to","the","park"]
for word in sentence:
  print(word)

the
boy
went
to
the
park


You can iterate over the keys and values of the dictionary that we created above as follows:

In [7]:
for key, value in thisdict.items():
    print(key + " " + str(value))

brand Ford
model Mustang
year 1964


### Bigram Counts

We are going to extract bigram counts from a text by using a for loop to iterate over our text, and a dictionary (specifically a defaultdict) to store the counts

First we prepare the text by tokenising it into a list of words. We also add a sentence boundary character "eol"

In [8]:
import re
from collections import defaultdict
import numpy as np
import pandas as pd
# download from from the internet
!wget https://www.gutenberg.org/files/2554/2554-0.txt
# read in the file
f = open('2554-0.txt')
c_and_p = f.read()
# Remove the title page etc
# convert text to lower case
c_and_p=c_and_p[5464:]
c_and_p=c_and_p.lower()
c_and_p=re.sub('\n',' ', c_and_p)
# Add end of sentence token
c_and_p=re.sub("\. "," eol ", c_and_p)
c_and_p=re.sub('[^a-z ]','', c_and_p)
c_and_p=re.sub(' +', ' ',c_and_p)
c_and_p=re.split(" ", c_and_p)


<>:16: SyntaxWarning: invalid escape sequence '\.'
<>:16: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-284156640.py:16: SyntaxWarning: invalid escape sequence '\.'
  c_and_p=re.sub("\. "," eol ", c_and_p)


--2026-02-24 16:19:48--  https://www.gutenberg.org/files/2554/2554-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1159923 (1.1M) [text/plain]
Saving to: ‘2554-0.txt’

2554-0.txt          100%[===================>]   1.11M  --.-KB/s    in 0.1s    

2026-02-24 16:19:49 (9.51 MB/s) - ‘2554-0.txt’ saved [1159923/1159923]



In [9]:
c_and_p

['some',
 'time',
 'past',
 'he',
 'had',
 'been',
 'in',
 'an',
 'overstrained',
 'irritable',
 'condition',
 'verging',
 'on',
 'hypochondria',
 'eol',
 'he',
 'had',
 'become',
 'so',
 'completely',
 'absorbed',
 'in',
 'himself',
 'and',
 'isolated',
 'from',
 'his',
 'fellows',
 'that',
 'he',
 'dreaded',
 'meeting',
 'not',
 'only',
 'his',
 'landlady',
 'but',
 'anyone',
 'at',
 'all',
 'eol',
 'he',
 'was',
 'crushed',
 'by',
 'poverty',
 'but',
 'the',
 'anxieties',
 'of',
 'his',
 'position',
 'had',
 'of',
 'late',
 'ceased',
 'to',
 'weigh',
 'upon',
 'him',
 'eol',
 'he',
 'had',
 'given',
 'up',
 'attending',
 'to',
 'matters',
 'of',
 'practical',
 'importance',
 'he',
 'had',
 'lost',
 'all',
 'desire',
 'to',
 'do',
 'so',
 'eol',
 'nothing',
 'that',
 'any',
 'landlady',
 'could',
 'do',
 'had',
 'a',
 'real',
 'terror',
 'for',
 'him',
 'eol',
 'but',
 'to',
 'be',
 'stopped',
 'on',
 'the',
 'stairs',
 'to',
 'be',
 'forced',
 'to',
 'listen',
 'to',
 'her',
 'trivi

Then we iterate over the text to extract bigram counts

In [11]:
from collections import defaultdict
total_unigrams = len(c_and_p) - 1
bigrams = defaultdict(int)
unigrams = defaultdict(int)
for i in range(total_unigrams-2):
    unigrams[c_and_p[i]] += 1
    bigrams[str.join(" ",c_and_p[i:i+2])] += 1

In [12]:
unigrams['motorcar']

0

In [13]:
unigrams["the"]/len(c_and_p)

0.03645588649816781

In [14]:
bigrams["the man"]/unigrams["the"]

0.008505154639175257

In [15]:
bigrams["in a"]/unigrams["in"]

0.09648845302119582

### Calculating the probabilities of sentences
We can then use the chain rule with a Markov assumption in order to calculate the probability of sentences:

# $p(the\ dog\ runs) = p( the| eol ) * p(dog|the) * p(runs|dog)$

In order to deal with unseen bigrams we will use add-one smoothing

In [16]:
sentence="the man came out"
sentence=sentence.split()
sentence.insert(0,"eol")
pr=1
for i in range(len(sentence)-1):
    ugr = sentence[i]
    bgr = sentence[i] + " " + sentence[i+1]
    pr *= (bigrams[bgr]+1)/(unigrams[ugr]+len(unigrams))
format(pr, '.50f')

'0.00000000001447366290831028698994132177909130573465'

Problem 1a: See what the highest probability 5 word sentence you can come up with is.

Problem 1b: See what the lowest probability 5 word sentence you can come up with is.

# Generation with language models

Just as we can calculate the probability of a known string using the chain rule and Markov assumption, we can also incrementally generate sentences via a "Markov process". The probability of any sentence being generated can be calculated using chain rule.

# $p(the\ dog\ runs) = p( the| eol ) * p(dog|the) * p(runs|dog)$

![decoding](https://raw.githubusercontent.com/cbannard/compling24/refs/heads/main/images/decoding.png)

### Greedy Search

In order to incrementally generate sentences the most obvious way to proceed is just to output the most probable word at each step.

![greedy](https://raw.githubusercontent.com/cbannard/compling24/refs/heads/main/images/greedysearch.png)

### While loops
In generating sentences we are going to make use of a second very useful type of loop - the while loop. This allows us to repeatedly perform some operation while a particular statement is true. For example

In [21]:
i=0
while i < 10:
  i=i+1
  print(i)

1
2
3
4
5
6
7
8
9
10


Before we start generating we are going to convert our bigrams counts into probabilities for convenience.

In [24]:
nested_dict = lambda: defaultdict(nested_dict)
d = nested_dict()

for bg in bigrams:
  ug = bg.split()
  # Removed print(bg) to avoid excessive output as it is not essential for the calculation

  # Check to prevent ZeroDivisionError if unigrams[ug[0]] is 0
  if unigrams[ug[0]] != 0:
    d[ug[1]][ug[0]] = bigrams[bg]/unigrams[ug[0]]
  else:
    # If the unigram count is zero, the probability of any word following it is zero.
    d[ug[1]][ug[0]] = 0.0 # Assign a float zero to maintain consistency

lm=pd.DataFrame(d)
lm

,time,past,he,had,been,in,an,overstrained,irritable,condition,...,gradual,renewal,regeneration,initiation,gutenberg,zzzzz,xxxxx,yyyyy,bbbbb,eats
some,0.049751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
from,0.006831,NaN,NaN,NaN,NaN,NaN,0.004098,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
to,0.001146,NaN,0.001146,NaN,NaN,0.000382,0.002102,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
any,0.014184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
the,0.009021,0.000773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000258,...,0.000129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nomads,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gradual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.5,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zzzzz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
xxxxx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


Try greedy decoding by running the code below with different starting words. What do you notice?

In [25]:
# Define starting word
w="i"
# Define stopping point - here when an end of line character is output
while w != "eol":
  print(w,end=' ')
  # get probabilities for all words following the previous word
  s=lm.loc[w]
  # sort the probabilities and output the most likely word
  w=s.sort_values(ascending=False).index[0]


i am not to the door 

### Sampling

Another way to pick words to output is to randomly choose them, but weighted by their probability in context. The probability then is the proportion of runs for which that word will be chosen.

![sampling](https://raw.githubusercontent.com/cbannard/compling24/refs/heads/main/images/sampling.png)

In [26]:
import numpy as np
# Specify starting word
w="he"
# Define stopping point - here when an end of line character is output
while w != "eol":
  print(w,end=' ')
  # get probabilities for all words following the previous word
  s=lm.loc[w]
  s=s.drop(s[np.isnan(s)].index)
  # Choose randomly from the probability distribution over next words
  w=np.random.choice(list(s.index),1,list(np.exp(s.values)))[0]

he dropped 

What problems do you notice with these sampled sentences?

### Generating with GPT

The problems that you see with the generated output above isn't just a result of the simple bigram model used to generate the probabilities. The same problems apply even when using a more sophisticated language model such as GPT.

In [1]:
!pip install matplotlib-venn
!apt-get -qq install -y libfluidsynth1

E: Package 'libfluidsynth1' has no installation candidate


In [2]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.5_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.5) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.5) ...
Processing triggers for man-db (2.10.2-1) ...
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00
  Created wheel for libarchive: filename=libarchive-0.4.7-py3-none-any.whl size=31629 sha256=b25213c5b7f1cd7d551af8f9b445333729f99302f7b3919f9182ecec2abd78a1
  Stored in directory: /root/.cache/pip/wheels/29/20/ab/f101da7b245b996aa097685ef742243725ea6150f5b3b6d9ed
Successfully built libarchive


In [3]:
# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

In [4]:
!pip install cartopy
import cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 49.0 MB/s eta 0:00:00


In [2]:
!pip uninstall -y tensorflow transformers
!pip install -q tensorflow==2.19.0
!pip install -q transformers
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# add the EOS token as PAD token to avoid warnings
model = TFGPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


ModuleNotFoundError: No module named 'transformers'

### Greedy Search Generation with GPT-2

In [30]:
# encode context the generation is conditioned on
input_ids = tokenizer.encode('I enjoy walking with my cute dog', return_tensors='tf')

# generate text until the output length (which includes the context length) reaches 50
greedy_output = model.generate(input_ids, max_length=50)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

NameError: name 'tokenizer' is not defined

### Pure sampling Generation with GPT-2

In [31]:
# set seed to reproduce results. Feel free to change the seed though to get different results
tf.random.set_seed(0)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=50,
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

NameError: name 'model' is not defined

# Top-k sampling

As a response to the problems seen with these sampling methods, researchers have come up with different methods. An example is top-K sampling, which was used in the released GPT-2 model. In this method we sample, but we do so from the top-K words rather than all the vocabulary. This excludes the less likely words that meant the texts were not meaningful.

![top_k_sampling](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/top_k_sampling.png)


In [32]:
# set seed to reproduce results. Feel free to change the seed though to get different results
tf.random.set_seed(0)

# set top_k to 50
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=50,
    top_k=50
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

NameError: name 'model' is not defined

# Task
```python
def calculate_sentence_probability(sentence_str):
    """
    Calculates the probability of a given sentence using bigram counts
    and add-one smoothing.

    Args:
        sentence_str (str): The sentence as a string of words separated by spaces.

    Returns:
        float: The calculated probability of the sentence.
    """
    sentence_words = sentence_str.split()
    # Prepend 'eol' (end of line) token to model the start of the sentence probability
    sentence_words.insert(0, "eol")
    
    probability = 1.0
    # Vocabulary size for add-one smoothing.
    # `unigrams` is a defaultdict, so `len(unigrams)` gives the number of unique word types seen.
    vocab_size = len(unigrams)

    # Iterate through the sentence to calculate the product of bigram probabilities
    for i in range(len(sentence_words) - 1):
        current_word = sentence_words[i]
        next_word = sentence_words[i+1]
        
        bigram_str = f"{current_word} {next_word}"
        
        # Get counts for the bigram and the unigram (current_word)
        # defaultdict(int) returns 0 if key not found, which is perfect for add-one smoothing.
        bigram_count = bigrams[bigram_str]
        unigram_count = unigrams[current_word]

        # Apply add-one smoothing formula:
        # P(next_word | current_word) = (C(current_word, next_word) + 1) / (C(current_word) + Vocabulary_Size)
        # Avoid division by zero if unigram_count + vocab_size is 0, though vocab_size should ensure it's not.
        denominator = unigram_count + vocab_size
        if denominator == 0:
            # This case should ideally not happen if vocab_size > 0.
            # If a word is completely unseen and vocab_size is 0, probability becomes 0.
            # However, with add-one smoothing, denominator is at least 1 (if vocab_size is 0,
            # which it shouldn't be, and unigram_count is 0, then 0+1=1 in numerator, 0+0=0 in denominator - this logic needs care).
            # Given that `unigrams` is populated from the corpus, `vocab_size` will be > 0.
            # Also, unigram_count is at least 0. So denominator will be at least `vocab_size`.
            # So, denominator will always be > 0.
            probability *= 0 # Effectively makes the sentence probability zero
        else:
            probability *= (bigram_count + 1) / denominator
    
    return probability

# --- Demonstration of function usage ---
print("Demonstrating `calculate_sentence_probability` function:\n")

# Sample 5-word sentence
sample_sentence_1 = "the man went to the"
prob_1 = calculate_sentence_probability(sample_sentence_1)
print(f"The probability of the sentence '{sample_sentence_1}' is: {format(prob_1, '.50f')}")

sample_sentence_2 = "i have no idea what"
prob_2 = calculate_sentence_probability(sample_sentence_2)
print(f"The probability of the sentence '{sample_sentence_2}' is: {format(prob_2, '.50f')}")

# --- Strategies for finding high-probability sentences ---
print("\n--- Strategies for finding high-probability and low-probability sentences ---")
print("To find a high-probability 5-word sentence, you could:")
print("1. **Test common phrases and sequences** directly from the corpus. Sentences made of frequently co-occurring words will have higher probabilities.")
print("2. **Utilize the `lm` (language model) DataFrame** created earlier. This DataFrame (after smoothing is applied to its values for true probabilities) shows the conditional probabilities `P(next_word | current_word)`. You can start with a common word and greedily pick the most probable next word for 4 subsequent steps.")
print("3. **Iterate through frequent unigrams** as starting points and build sentences using the most probable next words. Common starting words (e.g., 'the', 'i', 'he') are good candidates.")

print("\nConversely, to find a low-probability 5-word sentence, you would try sequences of words that are unlikely to appear together or contain very rare words from the corpus. For example, a sentence with words that never co-occur would result in a very low probability due to add-one smoothing (e.g., 'a zzzzz xxxxx yyyyy bbbbb').")
```

## Implement `calculate_sentence_probability` function

### Subtask:
Create a function that calculates the probability of a given sentence using bigram counts and add-one smoothing.


**Reasoning**:
To implement the `calculate_sentence_probability` function as per the instructions, I will define a Python function that takes a sentence string, tokenizes it, adds the 'eol' token, and then iterates through the bigrams to calculate the probability using add-one smoothing.



In [33]:
def calculate_sentence_probability(sentence_str):
    # Split the sentence into a list of words
    sentence_words = sentence_str.split()
    # Insert 'eol' at the beginning
    sentence_words.insert(0, "eol")

    # Initialize probability
    probability = 1.0

    # Get vocabulary size from the unigrams dictionary
    vocab_size = len(unigrams)

    # Iterate through the sentence words to form bigrams
    for i in range(len(sentence_words) - 1):
        current_word = sentence_words[i]
        next_word = sentence_words[i+1]

        bigram_str = f"{current_word} {next_word}"

        # Get counts with add-one smoothing
        bigram_count = bigrams[bigram_str]
        unigram_count = unigrams[current_word]

        # Apply add-one smoothing formula
        smoothed_probability = (bigram_count + 1) / (unigram_count + vocab_size)

        # Multiply to the total probability
        probability *= smoothed_probability

    return probability

print("The function `calculate_sentence_probability` has been defined.")

The function `calculate_sentence_probability` has been defined.


**Reasoning**:
Now that the `calculate_sentence_probability` function is defined, I will demonstrate its usage with sample sentences as provided in the problem description to verify its functionality.



In [18]:
print("Demonstrating `calculate_sentence_probability` function:\n")

# Sample 5-word sentence
sample_sentence_1 = "the man went to the"
prob_1 = calculate_sentence_probability(sample_sentence_1)
print(f"The probability of the sentence '{sample_sentence_1}' is: {format(prob_1, '.50f')}")

sample_sentence_2 = "i have no idea what"
prob_2 = calculate_sentence_probability(sample_sentence_2)
print(f"The probability of the sentence '{sample_sentence_2}' is: {format(prob_2, '.50f')}")

print("\n--- Strategies for finding high-probability and low-probability sentences ---")
print("To find a high-probability 5-word sentence, you could:")
print("1. **Test common phrases and sequences** directly from the corpus. Sentences made of frequently co-occurring words will have higher probabilities.")
print("2. **Utilize the `lm` (language model) DataFrame** created earlier. This DataFrame (after smoothing is applied to its values for true probabilities) shows the conditional probabilities `P(next_word | current_word)`. You can start with a common word and greedily pick the most probable next word for 4 subsequent steps.")
print("3. **Iterate through frequent unigrams** as starting points and build sentences using the most probable next words. Common starting words (e.g., 'the', 'i', 'he') are good candidates.")

print("\nConversely, to find a low-probability 5-word sentence, you would try sequences of words that are unlikely to appear together or contain very rare words from the corpus. For example, a sentence with words that never co-occur would result in a very low probability due to add-one smoothing (e.g., 'a zzzzz xxxxx yyyyy bbbbb').")

Demonstrating `calculate_sentence_probability` function:

The probability of the sentence 'the man went to the' is: 0.00000000000316141735318959931198726697777333128096
The probability of the sentence 'i have no idea what' is: 0.00000000000009870091110331079410344780290908999356

--- Strategies for finding high-probability and low-probability sentences ---
To find a high-probability 5-word sentence, you could:
1. **Test common phrases and sequences** directly from the corpus. Sentences made of frequently co-occurring words will have higher probabilities.
2. **Utilize the `lm` (language model) DataFrame** created earlier. This DataFrame (after smoothing is applied to its values for true probabilities) shows the conditional probabilities `P(next_word | current_word)`. You can start with a common word and greedily pick the most probable next word for 4 subsequent steps.
3. **Iterate through frequent unigrams** as starting points and build sentences using the most probable next words. Comm

## Problem 1a: Find highest probability 5-word sentence

### Subtask:
Identify a 5-word sentence from the corpus that has the highest probability according to the bigram model with add-one smoothing.


**Reasoning**:
As instructed, I will select a candidate 5-word sentence, 'he was not in the', and calculate its probability using the `calculate_sentence_probability` function that was previously defined. This will help in identifying a high-probability sentence.



In [19]:
print("\n--- Calculating probability for a new candidate sentence ---")
candidate_sentence_1 = "he was not in the"
prob_candidate_1 = calculate_sentence_probability(candidate_sentence_1)
print(f"The probability of the sentence '{candidate_sentence_1}' is: {format(prob_candidate_1, '.50f')}")

candidate_sentence_2 = "i do not know what"
prob_candidate_2 = calculate_sentence_probability(candidate_sentence_2)
print(f"The probability of the sentence '{candidate_sentence_2}' is: {format(prob_candidate_2, '.50f')}")

# For finding the highest probability, one would typically iterate through many candidate sentences
# or use a generation strategy like greedy search. Since the task is to 'find' one, we are checking candidates.
# To truly find the 'highest', a more exhaustive search would be needed or generation from the LM directly.


--- Calculating probability for a new candidate sentence ---
The probability of the sentence 'he was not in the' is: 0.00000000137775001676495722542288940274009256814480
The probability of the sentence 'i do not know what' is: 0.00000000000225512020078023155140131088814746618482


### Answer to Problem 1a: Highest Probability 5-Word Sentence

Among the candidate 5-word sentences evaluated, the sentence `'he was not in the'` yielded the highest probability using the bigram model with add-one smoothing. Its calculated probability is `0.00000000137775001676495722542288940274009256814480`.

**Reasoning**:
To find a low-probability 5-word sentence, I will construct a sentence using words that are highly unlikely to appear together or might not even be present in the corpus, and then calculate its probability using the `calculate_sentence_probability` function.



In [20]:
print("\n--- Calculating probability for a low-probability candidate sentence ---")
# Construct a sentence with unlikely word combinations or rare words
low_prob_sentence = "a zzzzz xxxxx yyyyy bbbbb"
prob_low = calculate_sentence_probability(low_prob_sentence)
print(f"The probability of the sentence '{low_prob_sentence}' is: {format(prob_low, '.50f')}")

# Another candidate with seemingly random common words, but unlikely sequence
low_prob_sentence_2 = "the tree eats green sky"
prob_low_2 = calculate_sentence_probability(low_prob_sentence_2)
print(f"The probability of the sentence '{low_prob_sentence_2}' is: {format(prob_low_2, '.50f')}")

# The lowest probability will be extremely small, likely due to zero counts for unigrams/bigrams
# which become 1/vocab_size * 1/vocab_size ... due to add-one smoothing.



--- Calculating probability for a low-probability candidate sentence ---
The probability of the sentence 'a zzzzz xxxxx yyyyy bbbbb' is: 0.00000000000000000039772615224270345738222730981251
The probability of the sentence 'the tree eats green sky' is: 0.00000000000000000106969455459760825526216006037230


### Answer to Problem 1b: Lowest Probability 5-Word Sentence

Among the candidate 5-word sentences evaluated, the sentence `'a zzzzz xxxxx yyyyy bbbbb'` yielded the lowest probability using the bigram model with add-one smoothing. Its calculated probability is `0.00000000000000000039772615224270345738222730981251`. This extremely low probability is expected because the words 'zzzzz', 'xxxxx', 'yyyyy', and 'bbbbb' are not present in the corpus, leading to probabilities based solely on the add-one smoothing factor for each bigram.

### Answer to Problem 1b: Lowest Probability 5-Word Sentence

Among the candidate 5-word sentences evaluated, the sentence `'a zzzzz xxxxx yyyyy bbbbb'` yielded the lowest probability using the bigram model with add-one smoothing. Its calculated probability is `0.00000000000000000039772615224270345738222730981251`. This extremely low probability is expected because the words 'zzzzz', 'xxxxx', 'yyyyy', and 'bbbbb' are not present in the corpus, leading to probabilities based solely on the add-one smoothing factor for each bigram.

### Answer to Problem 1b: Lowest Probability 5-Word Sentence

Among the candidate 5-word sentences evaluated, the sentence `'a zzzzz xxxxx yyyyy bbbbb'` yielded the lowest probability using the bigram model with add-one smoothing. Its calculated probability is `0.00000000000000000039772615224270345738222730981251`. This extremely low probability is expected because the words 'zzzzz', 'xxxxx', 'yyyyy', and 'bbbbb' are not present in the corpus, leading to probabilities based solely on the add-one smoothing factor for each bigram.

## Summary:

### Q&A
The highest probability 5-word sentence identified among the evaluated candidates, using the bigram model with add-one smoothing, is "he was not in the".

### Data Analysis Key Findings
*   The `calculate_sentence_probability` function was successfully implemented, computing bigram probabilities for sentences using add-one smoothing, and prepending an 'eol' token to model the start of sentences.
*   For a sample sentence "the man went to the", the calculated probability was approximately $3.16 \times 10^{-12}$.
*   For another sample sentence "i have no idea what", the calculated probability was approximately $9.87 \times 10^{-14}$.
*   Among the candidate sentences tested, "he was not in the" yielded the highest probability at approximately $1.38 \times 10^{-9}$.
*   Conversely, a sentence containing out-of-vocabulary words, "a zzzzz xxxxx yyyyy bbbbb", resulted in an extremely low probability of approximately $3.98 \times 10^{-19}$, demonstrating the impact of add-one smoothing on entirely unseen word sequences.

### Insights or Next Steps
*   To definitively find the highest probability 5-word sentence from the corpus, a more exhaustive search or a generative approach (e.g., greedy search starting from frequent unigrams and iteratively selecting the most probable next word) would be necessary, rather than testing a limited number of candidates.
*   The significant difference in probabilities between common phrases and sequences with rare or non-existent words effectively demonstrates how add-one smoothing addresses the zero-frequency problem while still assigning very low probabilities to highly improbable sequences.


# Task
Update the `!pip install` command to install a compatible and available version of TensorFlow to resolve the installation error and enable the `TFGPT2LMHeadModel` import.

## Update TensorFlow installation

### Subtask:
Modify the `!pip install` command to install a compatible and available version of TensorFlow to resolve the installation error and enable the `TFGPT2LMHeadModel` import.


## Summary:

### Data Analysis Key Findings
*   The solving process details for updating the TensorFlow installation were not provided, therefore no specific analysis findings or results can be reported.

### Insights or Next Steps
*   To generate insights or propose next steps, the actual execution steps and outcomes of the TensorFlow installation update are required.
